# YOLO11 žibintų detektorius (Google Colab)

Antrasis YOLO bandymas: modelis mokomas atpažinti priešpriešinių automobilių **žibintus** (ne kėbulus), kad matytų ir tolimus automobilius tamsiame užmiesčio kelyje.

Duomenys: Roboflow Universe rinkinys **„Automatic headlight dimming control“** (`amilyaj-gmail-com/automatic-headlight-dimming-control`, v2, 215 vaizdų: 152 mokymui, 42 validavimui, 21 testui).

**Prieš pradedant:**
1. *Execution environment (Runtime) → Change runtime type → T4 GPU → Save*.
2. Žemiau įklijuokite Roboflow API raktą.
3. *Run everything (Run all)*. Viskas trunka ~10–15 min, pabaigoje atsisiunčiamas `best_headlight.onnx`.

In [ ]:
!pip install -q ultralytics roboflow
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Duomenys

In [ ]:
API_KEY = ""   # <- čia įklijuokite Roboflow API raktą

import glob, os, shutil, yaml
from roboflow import Roboflow
rf = Roboflow(api_key=API_KEY)
ds = rf.workspace("amilyaj-gmail-com").project("automatic-headlight-dimming-control").version(2).download("yolov8")
root = ds.location

cfg = yaml.safe_load(open(f"{root}/data.yaml"))
print("KLASĖS:", cfg["names"])
print("Vaizdų:", {s: len(glob.glob(f"{root}/{s}/images/*")) for s in ("train", "valid", "test")})
cfg.update(path=root, train="train/images", val="valid/images", test="test/images")
yaml.safe_dump(cfg, open(f"{root}/data_abs.yaml", "w"))
DATA = f"{root}/data_abs.yaml"

## 2. Mokymas

Perkeliamasis mokymas nuo COCO išmokyto `yolo11n.pt`. Horizontalus apvertimas išjungtas (eismo pusė svarbi). Rinkinys mažas, todėl 80 epochų trunka ~5–10 min.

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")
model.train(data=DATA, epochs=80, imgsz=640, batch=16, fliplr=0.0, hsv_v=0.2,
            patience=20, project="/content/runs", name="headlight", exist_ok=True)

## 3. Patikrinimas su atskira testo dalimi (21 vaizdas, modelis jų nematė)

In [ ]:
best_pt = max(glob.glob("/content/**/headlight/weights/best.pt", recursive=True), key=os.path.getmtime)
best = YOLO(best_pt)
m = best.val(data=DATA, split="test")
print(f"TESTAS: mAP50 = {m.box.map50:.3f}   mAP50-95 = {m.box.map:.3f}")

## 4. Eksportas į ONNX ir atsisiuntimas

In [ ]:
path = best.export(format="onnx", imgsz=640, opset=12, simplify=True, dynamic=False)
shutil.copy(path, "best_headlight.onnx")
print("KLASĖS MODELYJE:", best.names)
from google.colab import files
files.download("best_headlight.onnx")

## Ką atsiųsti

1. Eilutę `KLASĖS MODELYJE: {...}`
2. Eilutę `TESTAS: mAP50 = ...`
3. Failą `best_headlight.onnx`